# QC Label Tambang Papua — Tang & Werner saja vs UNION dengan Maus dkk. 2022

**Latar belakang.** `evaluate()` model 1 (attention U-Net, 80 epoch) di test set Papua-holdout
menghasilkan Tambang IoU=0,19 walau recall=0,91 — 195 ribu piksel bukan-Tambang dituduh Tambang
(FP 4x lipat dari TP). `label_fusion.py` SUDAH punya union Tang & Werner 2023 + Maus dkk. 2022
(ditambahkan 9 Juni) untuk memperlebar cakupan footprint tambang. **TAPI** patch Papua
(`ForestWatch_Patches`, dipakai val/test) sudah dipotong SEBELUM penambahan itu — kemungkinan
besar label Tambang di test set masih murni Tang & Werner, belum ter-refresh union Maus.

**Tujuan notebook ini**: HANYA mengecek (bukan re-export/re-train) — hitung piksel Tambang di
bbox Papua dengan (a) Tang & Werner saja [kondisi label saat ini] vs (b) union +Maus [kondisi
kode `build_label()` sekarang] — supaya tahu pasti apakah test set perlu di-refresh labelnya
sebelum kita putuskan fine-tune loss-function (Seesaw/cRT) untuk masalah over-predict Tambang.

**Tidak menyentuh**: citra, patch yang sudah dipotong, checkpoint, atau training apa pun.
Murni `reduceRegion` ringan terhadap 2 FeatureCollection poligon.

In [5]:
# === SETUP (Colab) — clone repo + install ===
import sys, subprocess, importlib

subprocess.run(
    "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
    "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
    shell=True, check=False,
)
subprocess.run("pip install -q -e /content/fw_repo[gee]", shell=True, check=False)
if "/content/fw_repo/src" not in sys.path:
    sys.path.insert(0, "/content/fw_repo/src")
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
print("Setup selesai.")

Setup selesai.


In [6]:
# === AUTH GEE + DEFINISI WILAYAH PAPUA ===
import ee
from forestwatch.gee.auth import init_ee
from forestwatch.constants import GEE_ASSETS, PAPUA_BBOX
from forestwatch.config import load_config

cfg = load_config()
init_ee(project=cfg["project"]["gee_project_id"])  # forestwatch-papua-unand

papua = ee.Geometry.Rectangle(list(PAPUA_BBOX))
print(f"Wilayah: Papua bbox {PAPUA_BBOX}")
print(f"  mining (Tang & Werner) : {GEE_ASSETS['mining']}")
print(f"  mining_maus            : {GEE_ASSETS['mining_maus']}")

2026-06-21 15:32:11 [INFO] forestwatch.gee.auth: GEE siap (project=forestwatch-papua-unand).
Wilayah: Papua bbox (130.0, -9.5, 141.2, 0.5)
  mining (Tang & Werner) : projects/sat-io/open-datasets/global-mining/global_mining_footprints
  mining_maus            : projects/sat-io/open-datasets/global-mining/global_mining_polygons


In [7]:
# === HITUNG LUAS POLIGON TAMBANG: TW-saja vs UNION (+Maus) di bbox Papua ===
# Pakai area poligon langsung (Reducer.sum di atas raster paint scale 10m
# terlalu berat utk bbox sebesar Papua) -- ini cukup utk estimasi cepat,
# konsisten dgn cara build_label() membentuk is_mining (paint -> unmask 0).
mining_tw = ee.FeatureCollection(GEE_ASSETS["mining"]).filterBounds(papua)
mining_maus = ee.FeatureCollection(GEE_ASSETS["mining_maus"]).filterBounds(papua)
mining_union = mining_tw.merge(mining_maus)

n_tw = mining_tw.size().getInfo()
n_maus = mining_maus.size().getInfo()
area_tw_ha = mining_tw.geometry().area().divide(10000).getInfo()
area_union_ha = mining_union.geometry().dissolve().area().divide(10000).getInfo()

print(f"Poligon Tang & Werner di Papua : {n_tw} poligon, ~{area_tw_ha:,.1f} ha")
print(f"Poligon Maus di Papua          : {n_maus} poligon")
print(f"UNION (dissolve) di Papua      : ~{area_union_ha:,.1f} ha")
delta_ha = area_union_ha - area_tw_ha
pct = (delta_ha / area_tw_ha * 100) if area_tw_ha > 0 else float("inf")
print(f"\nSelisih (area yang BELUM ada di label test set sekarang): ~{delta_ha:,.1f} ha (+{pct:.1f}%)")

Poligon Tang & Werner di Papua : 18 poligon, ~4,202.4 ha
Poligon Maus di Papua          : 9 poligon
UNION (dissolve) di Papua      : ~5,830.8 ha

Selisih (area yang BELUM ada di label test set sekarang): ~1,628.3 ha (+38.7%)


In [9]:
# === ESTIMASI PIKSEL @10m -- DERIVASI ARITMATIKA dari area (TANPA panggil GEE lagi) ===
# reduceRegion raster (baik di seluruh bbox Papua maupun bounding box poligon tambang)
# selalu timeout: lokasi tambang tersebar jauh (Freeport di pegunungan tengah, Raja Ampat
# di barat, dll) jadi bounding box-nya tetap seluas Papua. Daripada paksa rasterisasi,
# cukup derivasi dari AREA (cell sebelumnya, vektor murni, sudah berhasil & cepat):
# 1 piksel @10m = 100 m^2 = 0.01 ha -> n_piksel = area_ha / 0.01 = area_ha * 100.
n_tw_px = area_tw_ha * 100
n_union_px = area_union_ha * 100
gap = n_union_px - n_tw_px

print('Estimasi piksel Tambang @10m (derivasi dari area poligon, bukan reduceRegion raster):')
print()
print('Tang & Werner saja (label SAAT INI)  :', format(n_tw_px, ',.0f'), 'piksel')
print('UNION +Maus (build_label sekarang)   :', format(n_union_px, ',.0f'), 'piksel')
print('Piksel TAMBAHAN dari Maus            :', format(gap, ',.0f'),
      'piksel (+{:.1f}%)'.format(gap / max(n_tw_px, 1) * 100))
print()
print('Catatan: estimasi dari area poligon, bukan rasterisasi asli -- cukup akurat utk QC ini.')


Estimasi piksel Tambang @10m (derivasi dari area poligon, bukan reduceRegion raster):

Tang & Werner saja (label SAAT INI)  : 420,243 piksel
UNION +Maus (build_label sekarang)   : 583,075 piksel
Piksel TAMBAHAN dari Maus            : 162,833 piksel (+38.7%)

Catatan: estimasi dari area poligon, bukan rasterisasi asli -- cukup akurat utk QC ini.


In [11]:
# === VISUALISASI: di mana saja piksel Tambang TAMBAHAN dari Maus (folium) ===
import folium

only_maus_extra = is_mining_union.And(is_mining_tw.Not())  # ada di union, tak ada di TW saja

# Basemap eksternal (OSM/CartoDB) gagal load di sandbox Colab -- background tetap hitam
# walau layer GEE (titik merah) berhasil tampil. Karena tile GEE TERBUKTI jalan di sesi
# ini, ganti basemap-nya jadi citra GEE juga (ESA WorldCover) -- lewat jalur yg sama,
# tak tergantung provider eksternal yg mungkin diblokir.
from forestwatch.constants import GEE_ASSETS as _ASSETS

basemap_img = ee.Image(_ASSETS["esa_worldcover"]).select("Map")
basemap_vis = {"min": 10, "max": 100,
               "palette": ["006400", "ffbb22", "ffff4c", "f096ff", "fa0000",
                           "b4b4b4", "f0f0f0", "0064c8", "0096a0", "00cf75", "fae6a0"]}

bounds_info = mining_union.geometry().bounds().coordinates().get(0).getInfo()
lons = [pt[0] for pt in bounds_info]
lats = [pt[1] for pt in bounds_info]

m = folium.Map(location=[sum(lats) / len(lats), sum(lons) / len(lons)], zoom_start=5,
                width=900, height=600, tiles=None)

base_map_id = basemap_img.getMapId(basemap_vis)
folium.TileLayer(
    tiles=base_map_id["tile_fetcher"].url_format, attr="Google Earth Engine (ESA WorldCover)",
    name="Basemap (ESA WorldCover)",
).add_to(m)

vis_extra = {"min": 0, "max": 1, "palette": ["00000000", "ff0000"]}
map_id = only_maus_extra.getMapId(vis_extra)
folium.TileLayer(
    tiles=map_id["tile_fetcher"].url_format, attr="Google Earth Engine",
    name="Tambang TAMBAHAN dari Maus (merah)", overlay=True,
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
m


## Kesimpulan & langkah berikut

- Kalau **selisih piksel/area di atas KECIL** (mendekati 0%): label test-set sekarang sudah
  representatif, gap Tambang murni masalah model (loss/training) — lanjut fine-tune Seesaw/cRT
  tanpa perlu re-export label.
- Kalau **selisih SIGNIFIKAN** (puluhan % atau lebih): sebagian dari 195 ribu piksel "false
  positive" Tambang di `metrics.json` kemungkinan BUKAN kesalahan model — label test-set-nya yang
  belum lengkap. Langkah lanjutan: re-export ULANG band `label` saja (bukan citra Sentinel-2,
  jauh lebih ringan) untuk tile Papua yang overlap dengan area merah di peta di atas, lalu
  re-cut HANYA patch yang terdampak, baru evaluasi ulang `metrics.json` test set.
- Peta folium di atas dipakai untuk cek manual: apakah area merah (tambahan Maus) jatuh di
  region yang termasuk split test/val (acak per-patch dari tile Papua), bukan cuma di train.

## Identifikasi tile Papua yang terdampak (untuk re-export label bertarget)

Patch training fisiknya ada sebagai TAR di `Bahan_Training_Fix/{train,val,test}/*.tar` (dicek
manual via Drive) -- bukan folder `.npz` lepas. Tiap `.npz` di dalam tar menyimpan field `tile`
(nama file `.tif` asal, mis. `papua_t2_tile_05.tif`), `row`, `col`. Supaya nanti bisa patch
`lab` IN-PLACE (tanpa re-generate split train/val/test yang tidak punya manifest per-file),
kita perlu tahu PERSIS index tile mana (dari grid 6x6, `papua_t2_tile_00` s.d. `_35`) yang
overlap poligon Maus -- bukan re-export seluruh 36 tile.

In [ ]:
# === TILE PAPUA MANA SAJA YANG OVERLAP POLIGON MAUS (utk re-export bertarget) ===
from forestwatch.gee.tiles import make_tiles_from_bbox

NX, NY = cfg['export']['tiles_nx'], cfg['export']['tiles_ny']  # 6 x 6 (sama dgn Bagian 7)
tile_bboxes = make_tiles_from_bbox(PAPUA_BBOX, nx=NX, ny=NY)
print('Total tile grid:', len(tile_bboxes), f'({NX}x{NY}), cek overlap dgn {n_maus} poligon Maus...')
print()

affected_tiles = []
for idx, (xmin, ymin, xmax, ymax) in enumerate(tile_bboxes):
    tile_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
    n_hit = mining_maus.filterBounds(tile_geom).size().getInfo()
    if n_hit > 0:
        tile_name = f'papua_t2_tile_{idx:02d}'
        affected_tiles.append(tile_name)
        bbox_str = f'{xmin:.2f},{ymin:.2f},{xmax:.2f},{ymax:.2f}'
        print(' [TERDAMPAK]', tile_name, '(idx=' + str(idx) + ',', n_hit, 'poligon Maus overlap, bbox=' + bbox_str + ')')

print()
print('Total tile terdampak:', len(affected_tiles), '/', len(tile_bboxes))
print('Daftar:', affected_tiles)
